In [1]:
from pyspark.sql.functions import col, count, countDistinct, sum as spark_sum, when

dq_results = []  # هنجمع فيها كل نتائج الفحوصات عشان نطبعها في تقرير آخر الكود

def log_check(check_name, passed, details=""):
    status = " PASS" if passed else " FAIL"
    dq_results.append((check_name, status, details))
    print(f"{status} | {check_name} | {details}")

StatementMeta(, ccd84182-f278-4519-91c0-d5e7e48ce263, 3, Finished, Available, Finished, False)

### Duplicate Keys (Grain Integrity)

In [2]:
print("="*70)
print(" Checking for Grain duplicates (Primary/Composite Keys)")
print("="*70)

# fact_order_items: order_id + order_item_id لازم يكون Unique
total = spark.table("dwh.fact_order_items").count()
distinct = spark.table("dwh.fact_order_items").select("order_id", "order_item_id").distinct().count()
log_check("fact_order_items grain uniqueness", total == distinct, f"total={total}, distinct={distinct}")

# fact_payments: order_id + payment_sequential
total = spark.table("dwh.fact_payments").count()
distinct = spark.table("dwh.fact_payments").select("order_id", "payment_sequential").distinct().count()
log_check("fact_payments grain uniqueness", total == distinct, f"total={total}, distinct={distinct}")

# fact_orders: order_id لازم يكون Unique تمامًا (Accumulating Snapshot = صف واحد لكل أوردر)
total = spark.table("dwh.fact_orders").count()
distinct = spark.table("dwh.fact_orders").select("order_id").distinct().count()
log_check("fact_orders grain uniqueness", total == distinct, f"total={total}, distinct={distinct}")

# Dimensions: تأكد إن مفيش أكتر من is_current=true لنفس الـ Natural Key (مهم جدًا لـ SCD2)
for dim, nk in [("dim_customer", "customer_unique_id"), ("dim_seller", "seller_id"), ("dim_product", "product_id")]:
    dup_current = (spark.table(f"dwh.{dim}")
        .filter(col("is_current") == True)
        .groupBy(nk)
        .agg(count("*").alias("cnt"))
        .filter(col("cnt") > 1)
        .count()
    )
    log_check(f"{dim}: single is_current per {nk}", dup_current == 0, f"violations={dup_current}")

StatementMeta(, ccd84182-f278-4519-91c0-d5e7e48ce263, 4, Finished, Available, Finished, False)

 Checking for Grain duplicates (Primary/Composite Keys)
 PASS | fact_order_items grain uniqueness | total=112650, distinct=112650
 PASS | fact_payments grain uniqueness | total=103886, distinct=103886
 PASS | fact_orders grain uniqueness | total=99441, distinct=99441
 PASS | dim_customer: single is_current per customer_unique_id | violations=0
 PASS | dim_seller: single is_current per seller_id | violations=0
 PASS | dim_product: single is_current per product_id | violations=0


### Referential Integrity (Orphan Records)

In [4]:
print("="*70)
print(" Checking Referential Integrity (Orphan Foreign Keys)")
print("="*70)

def check_orphans(fact_table, fk_col, dim_table, dim_key):
    df_fact = spark.table(f"dwh.{fact_table}").alias("f")
    df_dim = spark.table(f"dwh.{dim_table}").alias("d")
    
    orphans = (df_fact
        .join(df_dim, col(f"f.{fk_col}") == col(f"d.{dim_key}"), "left_anti")
        .filter(col(f"f.{fk_col}").isNotNull())
        .count()
    )
    log_check(f"{fact_table}.{fk_col} -> {dim_table}", orphans == 0, f"orphan rows={orphans}")

check_orphans("fact_order_items", "customer_key", "dim_customer", "customer_key")
check_orphans("fact_order_items", "seller_key", "dim_seller", "seller_key")
check_orphans("fact_order_items", "product_key", "dim_product", "product_key")
check_orphans("fact_payments", "customer_key", "dim_customer", "customer_key")
check_orphans("fact_payments", "payment_type_key", "dim_payment_type", "payment_type_key")
check_orphans("fact_orders", "customer_key", "dim_customer", "customer_key")
check_orphans("fact_orders", "order_status_key", "dim_order_status", "order_status_key")

StatementMeta(, ccd84182-f278-4519-91c0-d5e7e48ce263, 6, Finished, Available, Finished, False)

 Checking Referential Integrity (Orphan Foreign Keys)
 PASS | fact_order_items.customer_key -> dim_customer | orphan rows=0
 PASS | fact_order_items.seller_key -> dim_seller | orphan rows=0
 PASS | fact_order_items.product_key -> dim_product | orphan rows=0
 PASS | fact_payments.customer_key -> dim_customer | orphan rows=0
 PASS | fact_payments.payment_type_key -> dim_payment_type | orphan rows=0
 PASS | fact_orders.customer_key -> dim_customer | orphan rows=0
 PASS | fact_orders.order_status_key -> dim_order_status | orphan rows=0


### Checking for NULLs in critical columns

In [5]:
print("="*70)
print(" Checking for NULLs in critical columns")
print("="*70)

critical_columns = {
    "fact_order_items": ["order_id", "order_item_id", "price", "freight_value"],
    "fact_payments": ["order_id", "payment_sequential", "payment_value"],
    "fact_orders": ["order_id", "order_status_key"],
    "dim_customer": ["customer_key", "customer_unique_id"],
    "dim_product": ["product_key", "product_id"],
}

for table, cols in critical_columns.items():
    df = spark.table(f"dwh.{table}")
    for c in cols:
        null_count = df.filter(col(c).isNull()).count()
        log_check(f"{table}.{c} NOT NULL", null_count == 0, f"null rows={null_count}")

StatementMeta(, ccd84182-f278-4519-91c0-d5e7e48ce263, 7, Finished, Available, Finished, False)

 Checking for NULLs in critical columns
 PASS | fact_order_items.order_id NOT NULL | null rows=0
 PASS | fact_order_items.order_item_id NOT NULL | null rows=0
 PASS | fact_order_items.price NOT NULL | null rows=0
 PASS | fact_order_items.freight_value NOT NULL | null rows=0
 PASS | fact_payments.order_id NOT NULL | null rows=0
 PASS | fact_payments.payment_sequential NOT NULL | null rows=0
 PASS | fact_payments.payment_value NOT NULL | null rows=0
 PASS | fact_orders.order_id NOT NULL | null rows=0
 PASS | fact_orders.order_status_key NOT NULL | null rows=0
 PASS | dim_customer.customer_key NOT NULL | null rows=0
 PASS | dim_customer.customer_unique_id NOT NULL | null rows=0
 PASS | dim_product.product_key NOT NULL | null rows=0
 PASS | dim_product.product_id NOT NULL | null rows=0


### Verifying Row Count Reconciliation (STG vs DWH)

In [6]:
print("="*70)
print(" Reconciling row counts between STG and DWH")
print("="*70)

stg_items = spark.table("stg.stg_order_items").count()
dwh_items = spark.table("dwh.fact_order_items").count()
log_check("order_items row count match", stg_items == dwh_items, f"stg={stg_items}, dwh={dwh_items}")

stg_payments = spark.table("stg.stg_order_payments").count()
dwh_payments = spark.table("dwh.fact_payments").count()
log_check("payments row count match", stg_payments == dwh_payments, f"stg={stg_payments}, dwh={dwh_payments}")

stg_orders = spark.table("stg.stg_orders").count()
dwh_orders = spark.table("dwh.fact_orders").count()
log_check("orders row count match", stg_orders == dwh_orders, f"stg={stg_orders}, dwh={dwh_orders}")

# فحص إضافي مهم: مجموع total_price في fact_orders لازم يطابق مجموع price في order_items
sum_items = spark.table("stg.stg_order_items").agg(spark_sum("price")).collect()[0][0]
sum_fact = spark.table("dwh.fact_orders").agg(spark_sum("total_price")).collect()[0][0]
diff = abs(float(sum_items or 0) - float(sum_fact or 0))
log_check("total_price aggregation accuracy", diff < 1.0, f"stg_sum={sum_items:.2f}, fact_sum={sum_fact:.2f}, diff={diff:.2f}")

StatementMeta(, ccd84182-f278-4519-91c0-d5e7e48ce263, 8, Finished, Available, Finished, False)

 Reconciling row counts between STG and DWH
 PASS | order_items row count match | stg=112650, dwh=112650
 PASS | payments row count match | stg=103886, dwh=103886
 PASS | orders row count match | stg=99441, dwh=99441
 PASS | total_price aggregation accuracy | stg_sum=13591643.70, fact_sum=13591643.70, diff=0.00


In [8]:
print("\n" + "="*70)
print(" Final Data Quality Report Summary")
print("="*70)

passed = sum(1 for r in dq_results if "PASS" in r[1])
failed = sum(1 for r in dq_results if "FAIL" in r[1])

print(f"Total Checks: {len(dq_results)} |  Passed: {passed} |  Failed: {failed}\n")

if failed > 0:
    print(" Failed Data Quality Checks:")
    for name, status, details in dq_results:
        if "FAIL" in status:
            print(f"   - {name}: {details}")
else:
    print(" All checks passed successfully")

StatementMeta(, ccd84182-f278-4519-91c0-d5e7e48ce263, 10, Finished, Available, Finished, False)


 Final Data Quality Report Summary
Total Checks: 30 |  Passed: 30 |  Failed: 0

 All checks passed successfully
